In [1]:
import pandas as pd
import numpy as np

'''
데이터 로드 + `제거
'''
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/'+ txt_dir, sep='|', dtype = str, engine='python')
    df.columns = [str(c).strip().strip("`") for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip("`")
    return df

In [2]:
month_str = '202208'
#month_str = '202207'
#month_str =  '202206'

In [3]:
'''
월별 데이터 로드 함수 -> 다른 월에도 재사용 가능
'''
def load_month(month_str):
    return(
        load_csv(f'1. 주택기본정보/jtbasicinfo_{month_str}.txt'),
        load_csv(f'3. 전세임대 예측시세/depo_hosiseinfo_{month_str}.txt'),
        load_csv(f'5. 월세임대 예측시세/rent_hosiseinfo_{month_str}.txt')
    )
df_주택기본정보, df_전세예측, df_월세예측 = load_month(month_str)

In [4]:
'''
수정본 : geopandas로 행정동과 법정동 매핑 with bind_dong_11_2025_2Q.shp (공통 지역명 매핑 데이터)
'''
import geopandas as gpd
from shapely.geometry import Point

gdf_hdong = gpd.read_file('bnd_dong_11_2025_2Q.shp')
gdf_hdong = gdf_hdong.to_crs(epsg = 4326)
gdf_hdong = gdf_hdong.rename(columns = {'ADM_CD' : 'hdong_code', 'ADM_NM':'hdong_name'})

df_주택기본정보['LNG'] = pd.to_numeric(df_주택기본정보['LNG'], errors = 'coerce')
df_주택기본정보['LAT'] = pd.to_numeric(df_주택기본정보['LAT'], errors = 'coerce')

gdf_bldg = gpd.GeoDataFrame(
    df_주택기본정보, 
    geometry = gpd.points_from_xy(df_주택기본정보['LNG'], df_주택기본정보['LAT']),
    crs = 'EPSG:4326')

In [5]:
print(gdf_hdong[gdf_hdong['hdong_name'].str.contains('상일|강일')][['hdong_code','hdong_name']])

    hdong_code hdong_name
423   11250750        강일동
424   11250760       상일1동
425   11250770       상일2동


In [6]:
gdf_joined = gpd.sjoin(gdf_bldg, gdf_hdong[['hdong_code','hdong_name','geometry']], how = 'left', predicate='within')

In [ ]:
# 행정동명 후처리 : 통폐합 반영
dong_rename = {'상일1동':'상일동', '상일2동':'강일동'}
code_rename = {'11250760':'11250760', # 상일1동 -> 상일동 (코드 유지)
               '11250770':'11250750'} # 상일2동 -> 강일동 코드로 통합

gdf_joined['hdong_code'] = gdf_joined['hdong_code'].replace(code_rename)
gdf_joined['hdong_name'] = gdf_joined['hdong_name'].replace(dong_rename)
               
print(len(gdf_bldg), len(gdf_joined), gdf_joined['hdong_code'].isna().mean())

100783 100783 0.0


In [8]:
df_주택기본정보 = pd.DataFrame(gdf_joined.drop(columns = 'geometry'))
# 이후부터 전세/월세도 같은 hdong_code로 매핑
# (전세/월세엔 위경도가 없으니 주택기본정보 기준으로 조인
df_전세예측 = df_전세예측.merge(df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'), on='PKCODE1', how='left')
df_월세예측 = df_월세예측.merge(df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'), on='PKCODE1', how='left')

In [9]:
df_주택기본정보['gu_name'] = df_주택기본정보['ADDRESS'].str.extract(r'서울특별시\s+(\S+구)')

gu_map_auto = (
    df_주택기본정보[['hdong_code','gu_name']]
    .dropna(subset = ['gu_name'])
    .drop_duplicates('hdong_code')
)

print(f'총 {len(gu_map_auto)}개 행정동 매핑')
print(gu_map_auto[gu_map_auto['gu_name'].isna()])
gu_map_auto

총 399개 행정동 매핑
Empty DataFrame
Columns: [hdong_code, gu_name]
Index: []


,hdong_code,gu_name
0,11010720,종로구
153,11010530,종로구
430,11010540,종로구
438,11010600,종로구
568,11010610,종로구
...,...,...
98536,11250650,강동구
98667,11250670,강동구
99553,11250610,강동구
99800,11250630,강동구


In [10]:
ref_year = 2022
LOW_Q = 0.2
old_threshold = 30

def prepare_price_ratio(df, price_col, dong_code_col, label):
    '''
    전세/월세 각각에 대해
    하위 분위 기준 계산
    저가여부 확인자 생성
    행정동별 저가 비율 및 건수 계산
    '''
    tmp = df.copy()
    tmp = tmp[tmp[price_col].notna()].copy()
    q = tmp[price_col].quantile(LOW_Q)
    # 저가 indicator
    tmp[f'is_low_{label}'] = (tmp[price_col] <= q).astype(int)

    # 동별 집계
    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_obs=(price_col, 'size'), low_ratio=(f'is_low_{label}', 'mean')).reset_index())

    agg = agg.rename(columns={'n_obs':f'n_{label}', 'low_ratio':f'low_ratio_{label}'})
    return agg, q


def prepare_old_ratio(df, build_year_col, dong_code_col):
    '''
    건축연도 기준 노후 비율 계산
    '''
    tmp = df.copy()
    tmp['SYDATE_parsed']= pd.to_datetime(tmp['SYDATE'], errors='coerce')
    tmp['build_year_final']=tmp['SYDATE_parsed'].dt.year
    
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp.loc[tmp['build_year_final']==0, 'build_year_final']=np.nan
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp['building_age'] = ref_year - tmp['build_year_final'].astype(int)
    tmp['is_old'] = (tmp['building_age'] >= old_threshold).astype(int)

    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_building=('building_age', 'size'), old_ratio=('is_old', 'mean')).reset_index())

    return agg

In [11]:
'''
PNU코드의 앞 10자리와 뒤에 법정동 코드까지 이어지는 10자리 합이 같은지 체크 -> 동일
'''
df_주택기본정보['PNU'] = df_주택기본정보['PNU'].astype(str).str.strip()
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_주택기본정보['SREG'] = df_주택기본정보['SREG'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['SEUB'] = df_주택기본정보['SEUB'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['bjdong_code_check'] = df_주택기본정보['SREG']+df_주택기본정보['SEUB']
mismatch = df_주택기본정보[df_주택기본정보['bjdong_code'] != df_주택기본정보['bjdong_code_check']]
print(len(mismatch))

0


In [12]:
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_전세예측['bjdong_code'] = df_전세예측['PNU'].str[:10]
df_월세예측['bjdong_code'] = df_월세예측['PNU'].str[:10]

# 행정동 코드 매핑 -> 위에서 완료

## 이제부터 집계 단위는 모두 hdong_code

In [13]:
dong_code_col = 'hdong_code'
price_col_depo = 'DEPO_PRED'
price_col_rent = 'PRED_RENT'
build_year_col = 'SYDATE'

df_주택기본정보['BUILDYEAR']  = pd.to_numeric(df_주택기본정보['BUILDYEAR'], errors='coerce')
df_전세예측[price_col_depo] = pd.to_numeric(df_전세예측[price_col_depo], errors='coerce')
df_월세예측[price_col_rent] = pd.to_numeric(df_월세예측[price_col_rent], errors='coerce')

## 5-1. 저가 주거지 비율

In [14]:
depo_agg, depo_q = prepare_price_ratio(df_전세예측, price_col_depo, dong_code_col, label='depo')
rent_agg, rent_q = prepare_price_ratio(df_월세예측, price_col_rent, dong_code_col, label='rent')

## 5-2. 노후도 비율

In [15]:
old_agg = prepare_old_ratio(df_주택기본정보, build_year_col, dong_code_col)

## 5-3. 연립-다세대 집적도(신규 추가)
: 총 대지 면적 추가 ver

In [16]:
# DJAREA 품질 진단 -> 단 하나의 결측도 없이 완벽
df_주택기본정보['DJAREA'] = pd.to_numeric(df_주택기본정보['DJAREA'], errors='coerce')

total = len(df_주택기본정보)

null_count = df_주택기본정보['DJAREA'].isna().sum()
zero_count = (df_주택기본정보['DJAREA']==0).sum()
valid_count = ((df_주택기본정보['DJAREA']>0)&df_주택기본정보['DJAREA'].notna()).sum()

print(f'전체 건물 수 : {total:,}')
print(f'결측 (NaN) : {null_count:,} ({null_count/total:.1%})')
print(f'0값 : {zero_count:,} ({zero_count/total:.1%})')
print(f'유효값 (>0) : {valid_count:,} ({valid_count/total:.1%})')
print()
print(df_주택기본정보['DJAREA'].describe())

전체 건물 수 : 100,783
결측 (NaN) : 0 (0.0%)
0값 : 0 (0.0%)
유효값 (>0) : 100,783 (100.0%)

count    100783.000000
mean        423.695322
std        2003.229247
min           6.200000
25%         186.000000
50%         244.000000
75%         334.000000
max      271936.000000
Name: DJAREA, dtype: float64


In [17]:
dong_area_check = (df_주택기본정보.groupby('hdong_code')
                   .apply(lambda x : pd.Series({
                       'total' : len(x),
                       'valid_area' : ((x['DJAREA']>0) & x['DJAREA'].notna()).sum(),
                       'valid_rate' : ((x['DJAREA']>0) & x['DJAREA'].notna()).mean()
                   }), include_groups = False).reset_index())
print(dong_area_check[dong_area_check['valid_rate']<0.7].sort_values('valid_rate'))

Empty DataFrame
Columns: [hdong_code, total, valid_area, valid_rate]
Index: []


In [18]:
df_주택기본정보['SEDECNT'] = pd.to_numeric(df_주택기본정보['SEDECNT'], errors = 'coerce')
df_주택기본정보['DJAREA'] = pd.to_numeric(df_주택기본정보['DJAREA'], errors = 'coerce')

density_agg = (
    df_주택기본정보.groupby('hdong_code', dropna=False).agg(
        n_building = ('PKCODE1', 'nunique'),
        total_units = ('SEDECNT', 'sum'),
        total_area = ('DJAREA', 'sum')).reset_index()) # 총 대지면적 추가

# 면적 대비 세대 밀도
density_agg['unit_density'] = density_agg['total_units']/density_agg['total_area']

In [19]:
print(density_agg.head())
print(density_agg['unit_density'].describe())

  hdong_code  n_building  total_units  total_area  unit_density
0   11010530          78          579    26119.07      0.022168
1   11010540           8           49     3219.31      0.015221
2   11010550         228         1535   297365.74      0.005162
3   11010560         368         2857  1163143.59      0.002456
4   11010570           9          123     3353.83      0.036674
count    399.000000
mean       0.027041
std        0.011203
min        0.000830
25%        0.021797
50%        0.028113
75%        0.031701
max        0.135672
Name: unit_density, dtype: float64


# 6. HVI score
## 6-1. 3개축 merge

In [20]:
hvi = pd.merge(depo_agg, rent_agg, on = 'hdong_code', how = 'outer')
hvi = pd.merge(hvi, old_agg, on = 'hdong_code', how = 'left')
hvi = pd.merge(hvi, density_agg, on = 'hdong_code', how = 'left')

hvi['n_rent']=hvi['n_rent'].fillna(0)
hvi['low_ratio_rent'] = hvi['low_ratio_rent'].fillna(np.nan)

In [21]:
def weighted_low_ratio(row):
    n_depo = row['n_depo']
    n_rent = row['n_rent']
    total = n_depo + n_rent
    if total == 0: return np.nan
    val = 0
    if pd.notna(row['low_ratio_depo']):
        val+= n_depo*row['low_ratio_depo']
    if pd.notna(row['low_ratio_rent']):
        val += n_rent*row['low_ratio_rent']
    return val/total

hvi['low_ratio_combined'] = hvi.apply(weighted_low_ratio, axis =1)

In [22]:
hvi['score_low'] = pd.qcut(hvi['low_ratio_combined'], q=5, labels=[1,2,3,4,5], duplicates = 'drop').astype(float)
hvi['score_old'] = pd.qcut(hvi['old_ratio'], q=5, labels=[1,2,3,4,5], duplicates = 'drop').astype(float)

# total_units 대신에 unit_density 사용
hvi['score_density'] = pd.qcut(hvi['unit_density'], q = 5, labels = [1,2,3,4,5], duplicates='drop').astype(float)

# 합산 (최대 15점)
hvi['HVI_score'] = hvi['score_low']+hvi['score_old']+hvi['score_density']

# 행정동 매핑
hdong_name_map = (df_주택기본정보[['hdong_code', 'hdong_name']].drop_duplicates('hdong_code'))
hvi = hvi.merge(hdong_name_map, on='hdong_code', how='left')

# 최소 표본 기준 설정
min_building = 5
min_obs = 10
hvi['n_total'] = hvi['n_depo'] + hvi['n_rent']
print(f'필터링 전: {len(hvi)}개 동')
hvi_filtered = hvi[(hvi['n_building_x']>=min_building) & (hvi['n_total']>=min_obs)].copy()
print(f'필터링 후: {len(hvi_filtered)}개 동')
print(f'제외된 동: {len(hvi)-len(hvi_filtered)}개')
excluded = hvi[(hvi['n_building_x']<min_building)|(hvi['n_total']<min_obs)][['hdong_name','n_building_x', 'n_total']]
print(excluded)


필터링 전: 393개 동
필터링 후: 388개 동
제외된 동: 5개
    hdong_name  n_building_x  n_total
147        창4동             4     56.0
153       월계3동             4     32.0
157       중계1동             1      8.0
163     중계2·3동             2     32.0
376        위례동             1     20.0


In [23]:
hvi = hvi_filtered.copy()

hvi['HVI_rank']=hvi['HVI_score'].rank(ascending=False, method='min')
hvi['HVI_grade'] = pd.qcut(hvi['HVI_score'], q=5, labels=['매우낮음','낮음','보통','높음','매우높음'])

## 6-3.Sanity Check
* HVI 상위 동 확인 -> 도봉 노원 중랑 관악 등 나오면 정상
* 강남 서초가 상위면 기준값 재검토
* 결측 비율 확인

In [24]:
# 1. HVI 상위 20개 동 확인
top20 = hvi.sort_values('HVI_rank').head(20)[['hdong_name', 'old_ratio','low_ratio_combined', 'unit_density', 'HVI_score', 'HVI_grade']]
print(top20.to_string())

# 2. 결측 비율
print('\n결측비율 : ')
print(hvi.isna().mean().sort_values(ascending=False))

# 3. 점수 분포
print('\nHVI_score 분포 : ')
print(hvi['HVI_score'].describe())

# 4. 등급별 동 수
print('\n등급별 동 수 : ')
print(hvi['HVI_grade'].value_counts())

    hdong_name  old_ratio  low_ratio_combined  unit_density  HVI_score HVI_grade
310        신원동   0.448598            0.354118      0.037311       15.0      매우높음
246       구로3동   0.360324            0.583978      0.042882       15.0      매우높음
219       신월6동   0.833333            0.868927      0.033385       15.0      매우높음
380       고덕2동   0.521277            0.391499      0.037620       15.0      매우높음
148        창5동   0.377907            0.455224      0.034062       15.0      매우높음
19         광희동   0.250000            0.331034      0.034775       14.0      매우높음
178       응암3동   0.385366            0.288871      0.033821       14.0      매우높음
125        번2동   0.712245            0.785050      0.030666       14.0      매우높음
379       고덕1동   0.739130            0.360465      0.030849       14.0      매우높음
34        이촌2동   0.868852            0.299113      0.033489       14.0      매우높음
131        미아동   0.384956            0.416097      0.031461       14.0      매우높음
113       길음2동   0.500000   

In [25]:
print(hvi[hvi['hdong_name']=='중계1동'][['n_depo','n_rent','n_building_x']])

Empty DataFrame
Columns: [n_depo, n_rent, n_building_x]
Index: []


In [26]:
hvi.head()

,hdong_code,n_depo,low_ratio_depo,n_rent,low_ratio_rent,n_building_x,old_ratio,n_building_y,total_units,total_area,unit_density,low_ratio_combined,score_low,score_old,score_density,HVI_score,hdong_name,n_total,HVI_rank,HVI_grade
0,11010530,318,0.122642,289.0,0.114187,78,0.192308,78,579,26119.07,0.022168,0.118616,2.0,3.0,2.0,7.0,사직동,607.0,276.0,매우낮음
1,11010540,5,0.000000,5.0,0.000000,8,0.000000,8,49,3219.31,0.015221,0.000000,1.0,1.0,1.0,3.0,삼청동,10.0,383.0,매우낮음
2,11010550,1627,0.083589,1584.0,0.055556,225,0.475556,228,1535,297365.74,0.005162,0.069760,1.0,5.0,1.0,7.0,부암동,3211.0,276.0,매우낮음
3,11010560,2373,0.038769,2350.0,0.020426,368,0.364130,368,2857,1163143.59,0.002456,0.029642,1.0,5.0,1.0,7.0,평창동,4723.0,276.0,매우낮음
4,11010570,47,0.148936,47.0,0.085106,9,0.444444,9,123,3353.83,0.036674,0.117021,2.0,5.0,5.0,12.0,무악동,94.0,30.0,매우높음


In [27]:
hvi = hvi.merge(gu_map_auto, on='hdong_code', how='left')
unmapped = hvi[hvi['gu_name'].isna()][['hdong_code','hdong_name']]
if len(unmapped)>0:
    print(f'경고 gu_name 매핑 실패 {len(unmapped)}개')
    print(unmapped)

In [28]:
hvi

,hdong_code,n_depo,low_ratio_depo,n_rent,low_ratio_rent,n_building_x,old_ratio,n_building_y,total_units,total_area,...,low_ratio_combined,score_low,score_old,score_density,HVI_score,hdong_name,n_total,HVI_rank,HVI_grade,gu_name
0,11010530,318,0.122642,289.0,0.114187,78,0.192308,78,579,26119.07,...,0.118616,2.0,3.0,2.0,7.0,사직동,607.0,276.0,매우낮음,종로구
1,11010540,5,0.000000,5.0,0.000000,8,0.000000,8,49,3219.31,...,0.000000,1.0,1.0,1.0,3.0,삼청동,10.0,383.0,매우낮음,종로구
2,11010550,1627,0.083589,1584.0,0.055556,225,0.475556,228,1535,297365.74,...,0.069760,1.0,5.0,1.0,7.0,부암동,3211.0,276.0,매우낮음,종로구
3,11010560,2373,0.038769,2350.0,0.020426,368,0.364130,368,2857,1163143.59,...,0.029642,1.0,5.0,1.0,7.0,평창동,4723.0,276.0,매우낮음,종로구
4,11010570,47,0.148936,47.0,0.085106,9,0.444444,9,123,3353.83,...,0.117021,2.0,5.0,5.0,12.0,무악동,94.0,30.0,매우높음,종로구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383,11250710,2316,0.118739,2130.0,0.144131,285,0.182456,285,2471,99083.80,...,0.130904,2.0,3.0,2.0,7.0,둔촌2동,4446.0,276.0,매우낮음,강동구
384,11250720,6190,0.079483,5733.0,0.117565,689,0.069666,689,6357,227676.02,...,0.097794,2.0,1.0,3.0,6.0,암사1동,11923.0,331.0,매우낮음,강동구
385,11250730,5105,0.061508,4585.0,0.086587,544,0.119485,544,5263,181906.89,...,0.073375,1.0,2.0,3.0,6.0,천호2동,9690.0,331.0,매우낮음,강동구
386,11250740,4844,0.091866,4621.0,0.125298,522,0.166667,522,5056,176915.96,...,0.108188,2.0,3.0,3.0,8.0,길동,9465.0,229.0,낮음,강동구


In [29]:
hvi['HVI_index'] = (
    (hvi['HVI_score'] - hvi['HVI_score'].min())/(hvi['HVI_score'].max() - hvi['HVI_score'].min())*100).round().astype(int)

export_hvi = hvi[['hdong_code','gu_name','hdong_name','low_ratio_combined', 'old_ratio', 'unit_density','score_low','score_old','score_density','HVI_score', 'HVI_index','HVI_rank', 'HVI_grade']].copy()
export_hvi.to_csv(f'HVI_bnd_dong_{month_str}_final.csv', index=False, encoding='utf-8-sig')